In [73]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_chroma import Chroma



In [74]:
llm = ChatOllama(model="qwen3-coder:30b")
embeddings = OllamaEmbeddings(model="nomic-embed-text:latest")

document = PyPDFLoader("/home/aakash/Projects/AI/interview-practice/src/docs/04-Policy-Manual-Apr-2016-excl-SAICA.pdf").load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

chunks = text_splitter.split_documents(documents=document)


for chunk in chunks:
    chunk.metadata.update({
        "version" : "3.1",
        "Role" : ["Admin", "HR"],
        "Document_id" : "HR-01",
    })
ids = [
    f"{chunk.metadata['Document_id']}_{i}"
    for i, chunk in enumerate(chunks)
]

print(ids)

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    ids=ids,
    collection_name="EmployeePolicy",
    persist_directory="./Employee_db"
)

fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/KLQRNP+Times-Roman', '/FontDescriptor': IndirectObject(10, 0, 138135836153424), '/Type': '/Font', '/FirstChar': 32, '/LastChar': 95, '/Widths': [250, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 500], '/Encoding': '/WinAnsiEncoding', '/Subtype': '/Type1'}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/ONXHWD+Helvetica-Bold', '/FontDescriptor': IndirectObject(12, 0, 138135836153424), '/Type': '/Font', '/FirstChar': 32, '/LastChar': 146, '/Widths': [278, 0, 0, 0, 0, 0, 722, 0, 333, 333, 0, 0, 278, 333, 278, 278, 556, 556, 556, 556, 556, 556, 556, 556, 556, 556, 333, 0, 0, 0, 0, 0, 0, 722, 722, 722, 722

['HR-01_0', 'HR-01_1', 'HR-01_2', 'HR-01_3', 'HR-01_4', 'HR-01_5', 'HR-01_6', 'HR-01_7', 'HR-01_8', 'HR-01_9', 'HR-01_10', 'HR-01_11', 'HR-01_12', 'HR-01_13', 'HR-01_14', 'HR-01_15', 'HR-01_16', 'HR-01_17', 'HR-01_18', 'HR-01_19', 'HR-01_20', 'HR-01_21', 'HR-01_22', 'HR-01_23', 'HR-01_24', 'HR-01_25', 'HR-01_26', 'HR-01_27', 'HR-01_28', 'HR-01_29', 'HR-01_30', 'HR-01_31', 'HR-01_32', 'HR-01_33', 'HR-01_34', 'HR-01_35', 'HR-01_36', 'HR-01_37', 'HR-01_38', 'HR-01_39', 'HR-01_40', 'HR-01_41', 'HR-01_42', 'HR-01_43', 'HR-01_44', 'HR-01_45', 'HR-01_46', 'HR-01_47', 'HR-01_48', 'HR-01_49', 'HR-01_50', 'HR-01_51', 'HR-01_52', 'HR-01_53', 'HR-01_54', 'HR-01_55', 'HR-01_56', 'HR-01_57', 'HR-01_58', 'HR-01_59', 'HR-01_60', 'HR-01_61', 'HR-01_62', 'HR-01_63', 'HR-01_64', 'HR-01_65', 'HR-01_66', 'HR-01_67', 'HR-01_68', 'HR-01_69', 'HR-01_70', 'HR-01_71', 'HR-01_72', 'HR-01_73', 'HR-01_74', 'HR-01_75', 'HR-01_76', 'HR-01_77', 'HR-01_78', 'HR-01_79', 'HR-01_80', 'HR-01_81', 'HR-01_82', 'HR-01_83', '

In [77]:
retriver = vector_store.as_retriever(kwargs=3)

query = "What is my notice period"
context = ""

docs = retriver.invoke(query)


for doc in docs:
    context += "\n" + doc.page_content

In [81]:
from langchain_core.prompts import PromptTemplate

def callLLM():

    prompt = PromptTemplate(template=f"""
    You are a helpfull Employee Policy answer generatore:
    Based on the following context, You answer the query.

    Rule: Do not generate answer if not available in the context.

    context: {context}
    query: {query}
    """ , input_variables=["context", "query"])

    chain = prompt | llm

    res = chain.invoke({
        "context": context,
        "query" : query
    })

    print(res.usage_metadata)

callLLM()

{'input_tokens': 443, 'output_tokens': 215, 'total_tokens': 658}
